In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path
import numpy as onp
import jax
import jax.numpy as jnp
from tqdm import tqdm

from msmjax.core.shortrange import make_eval_pair_pot
from msmjax.utils.benchmarking import (
    eval_lammps_pppm,
    path_input_structures,
)

In [2]:
LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

# Function definitions

In [3]:
def coulomb_kernel(r):
    return 1.0 / r


def calc_nonperiodic_ref_energy(positions, charges):
    n_dim = positions.shape[1]
    compute_pair_term = make_eval_pair_pot(
        kernel_fn=coulomb_kernel, pbc=(False,) * n_dim
    )
    return compute_pair_term(positions, charges)


def calc_nonperiodic_ref_forces(positions, charges):
    return -jax.grad(calc_nonperiodic_ref_energy, argnums=0)(
        positions, charges
    )


def calc_nonperiodic_ref_chargegrad(positions, charges):
    return jax.grad(calc_nonperiodic_ref_energy, argnums=1)(positions, charges)


def calc_energy_from_scaled(scaled_positions, charges, cell):
    positions = scaled_positions @ cell
    return calc_nonperiodic_ref_energy(positions, charges)


def calc_stress(positions, charges, cell):
    n_dim = positions.shape[1]
    scaled_positions = jnp.linalg.solve(cell.T, positions.T).T

    def deformation_energy(epsilon):
        return calc_energy_from_scaled(
            scaled_positions,
            charges,
            cell @ (jnp.eye(n_dim) + 0.5 * (epsilon + epsilon.T)),
        )

    return jax.grad(deformation_energy)(jnp.zeros_like(cell)) / jnp.fabs(
        jnp.linalg.det(cell)
    )


inds_matrix_to_six_component_stress = (
    jnp.array([0, 1, 2, 0, 0, 1]),
    jnp.array([0, 1, 2, 1, 2, 2]),
)


def calc_nonperiodic_reference_results(positions, charges, cell):
    # TODO: Put this function into utils? It is used both here and in
    #  tests/data/generate_reference_results.ipynb
    # The following is less likely to run out of memory than calculating
    # everything with a single jax.value_and_grad call
    energy = jax.jit(calc_nonperiodic_ref_energy)(positions, charges)
    forces = jax.jit(calc_nonperiodic_ref_forces)(positions, charges)
    chargegrad = jax.jit(calc_nonperiodic_ref_chargegrad)(positions, charges)
    stress = jax.jit(calc_stress)(positions, charges, cell)
    return energy, forces, chargegrad, stress

# Non-periodic

In [4]:
n_particles = 10000
outdir = Path("nonperiodic/")

outdir.mkdir(parents=True, exist_ok=True)
structures = onp.load(path_input_structures / f"structures_{n_particles}.npz")
n_structures = structures["positions"].shape[0]

all_energies = onp.full(n_structures, onp.nan)
all_forces = onp.full((n_structures, n_particles, 3), onp.nan)
all_chargegrads = onp.full((n_structures, n_particles), onp.nan)
all_stresses = onp.full((n_structures, 6), onp.nan)
for i in tqdm(range(n_structures)):
    pos = structures["positions"][i].astype(onp.float64)
    chg = structures["charges"][i].astype(onp.float64)
    cll = structures["cells"][i].astype(onp.float64)
    energy, forces, chargegrad, stress = calc_nonperiodic_reference_results(
        pos, chg, cll
    )
    all_energies[i] = energy
    all_forces[i] = forces
    all_chargegrads[i] = chargegrad
    all_stresses[i] = stress[inds_matrix_to_six_component_stress]

onp.savez_compressed(
    outdir / "structures.npz",
    positions=structures["positions"].astype(onp.float64),
    charges=structures["charges"].astype(onp.float64),
    cells=structures["cells"].astype(onp.float64),
)
onp.savez_compressed(
    outdir / "reference_results.npz",
    energies=all_energies,
    forces=all_forces,
    chargegrads=all_chargegrads,
    stresses=all_stresses,
)

100%|██████████| 11/11 [00:07<00:00,  1.43it/s]


In [72]:
@jax.jit
def cool_calc_stress_from_virial(r, f, cell):
    volume = jnp.linalg.det(cell)
    dotproducts = [
        jnp.dot(r[:, i], f[:, j])
        for i, j in zip(*inds_matrix_to_six_component_stress)
    ]
    return -jnp.array(dotproducts) / volume

In [73]:
cool_calc_stress_from_virial(pos, forces, cll)

Array([ 0.00482554, -0.01425329,  0.01317106,  0.0056601 , -0.00762442,
        0.0046813 ], dtype=float64)

In [74]:
all_stresses[-1]

array([-0.00659676,  0.01088426, -0.00043593, -0.00661704, -0.00069732,
       -0.00409493])

In [75]:
pos = jax.device_put(structures["positions"][0].astype(onp.float64))
chg = jax.device_put(structures["charges"][0].astype(onp.float64))
cll = jax.device_put(structures["cells"][0].astype(onp.float64))
forces = jax.device_put(all_forces[0])

In [76]:
@jax.jit
def maybe_cooler_calc_stress_from_virial(r, f, cell):
    volume = jnp.linalg.det(cell)
    (i, j) = inds_matrix_to_six_component_stress
    dotproducts = jax.vmap(jnp.dot, in_axes=(1, 1))(r[:, i], f[:, j])
    return -dotproducts / volume

In [77]:
cool_calc_stress_from_virial(pos, forces, cll).block_until_ready()
%timeit cool_stress_fn(pos, forces, cll).block_until_ready()

cool_calc_stress_from_virial(pos, forces, cll)

46.1 μs ± 2.64 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Array([ 0.00482554, -0.01425329,  0.01317106,  0.0056601 , -0.00762442,
        0.0046813 ], dtype=float64)

In [78]:
maybe_cooler_calc_stress_from_virial(pos, forces, cll).block_until_ready()
%timeit maybe_cooler_stress_fn(pos, forces, cll).block_until_ready()

maybe_cooler_calc_stress_from_virial(pos, forces, cll)

53.9 μs ± 9.51 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Array([ 0.00482554, -0.01425329,  0.01317106,  0.0056601 , -0.00762442,
        0.0046813 ], dtype=float64)

In [81]:
def maybe_cooler_calc_nonperiodic_reference_results(positions, charges, cell):
    # TODO: Put this function into utils? It is used both here and in
    #  tests/data/generate_reference_results.ipynb
    # The following is less likely to run out of memory than calculating
    # everything with a single jax.value_and_grad call
    energy = jax.jit(calc_nonperiodic_ref_energy)(positions, charges)
    forces = jax.jit(calc_nonperiodic_ref_forces)(positions, charges)
    chargegrad = jax.jit(calc_nonperiodic_ref_chargegrad)(positions, charges)
    stress = jax.jit(maybe_cooler_calc_stress_from_virial)(
        positions, forces, cell
    )
    return energy, forces, chargegrad, stress

In [83]:
calc_nonperiodic_reference_results(pos, chg, cll)[0].block_until_ready()

%timeit calc_nonperiodic_reference_results(pos, chg, cll)[0].block_until_ready()

569 ms ± 14.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [84]:
maybe_cooler_calc_nonperiodic_reference_results(pos, chg, cll)[
    0
].block_until_ready()

%timeit maybe_cooler_calc_nonperiodic_reference_results(pos, chg, cll)[0].block_until_ready()

302 ms ± 7.39 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# Periodic

In [5]:
n_particles = 2000
outdir = Path("periodic/")

outdir.mkdir(parents=True, exist_ok=True)
structures = onp.load(path_input_structures / f"structures_{n_particles}.npz")
n_structures = structures["positions"].shape[0]

all_energies = onp.full(n_structures, onp.nan)
all_forces = onp.full((n_structures, n_particles, 3), onp.nan)
all_chargegrads = onp.full((n_structures, n_particles), onp.nan)
all_stresses = onp.full((n_structures, 6), onp.nan)
for i in tqdm(range(n_structures)):
    pos = structures["positions"][i].astype(onp.float64)
    chg = structures["charges"][i].astype(onp.float64)
    cll = structures["cells"][i].astype(onp.float64)
    energy, forces, chargegrad, stress = eval_lammps_pppm(
        pos,
        chg,
        cll,
        LAMMPS_EXECUTABLE,
        accuracy=1.0e-8,
        max_neighbors_one_atom=10000,
    )
    all_energies[i] = energy
    all_forces[i] = forces
    all_chargegrads[i] = chargegrad
    all_stresses[i] = stress

onp.savez_compressed(
    outdir / "structures.npz",
    positions=structures["positions"].astype(onp.float64),
    charges=structures["charges"].astype(onp.float64),
    cells=structures["cells"].astype(onp.float64),
)
onp.savez_compressed(
    outdir / "reference_results.npz",
    energies=all_energies,
    forces=all_forces,
    chargegrads=all_chargegrads,
    stresses=all_stresses,
)

100%|██████████| 11/11 [00:04<00:00,  2.58it/s]
